# Autoencoder - Compression & Reconstruction

## الترتيب / Flow
1. استيراد المكتبات - Import libraries
2. قراءة البيانات - Load MNIST sample CSV
3. تجهيز الصور - Normalize and flatten
4. تقسيم البيانات - Train/Test split
5. بناء Autoencoder - Encoder + bottleneck + Decoder
6. compile + fit - Train with MSE reconstruction loss
7. إعادة البناء - Reconstruct test images
8. رسم original vs reconstructed - Visual comparison
9. فضاء latent - 2D projection sample

In [ ]:
# Step 1) استيراد المكتبات / Import libraries
# pip install tensorflow -q  # uncomment in Colab if needed
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

In [ ]:
# Step 2) قراءة البيانات / Load dataset
dataset = pd.read_csv('mnist_sample.csv')
print('Shape:', dataset.shape)
dataset.head()

In [ ]:
# Step 3) تجهيز الصور / Prepare and normalize images
pixel_cols = [c for c in dataset.columns if c.startswith('pixel_')]
X = dataset[pixel_cols].values.astype('float32') / 255.0
print('X shape:', X.shape)

In [ ]:
# Step 4) تقسيم البيانات / Train-Test split
X_train, X_test = train_test_split(X, test_size=0.2, random_state=0)
print('Train:', X_train.shape, '| Test:', X_test.shape)

In [ ]:
# Step 5) بناء Autoencoder / Encoder -> latent -> Decoder
input_dim = X.shape[1]
latent_dim = 32

inputs = Input(shape=(input_dim,))
encoded = Dense(128, activation='relu')(inputs)
encoded = Dense(64, activation='relu')(encoded)
latent = Dense(latent_dim, activation='relu', name='latent')(encoded)
decoded = Dense(64, activation='relu')(latent)
decoded = Dense(128, activation='relu')(decoded)
outputs = Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = Model(inputs, outputs)
encoder = Model(inputs, latent)
autoencoder.summary()

In [ ]:
# Step 6) compile + fit / Train autoencoder
autoencoder.compile(optimizer='adam', loss='mse')
history = autoencoder.fit(
    X_train, X_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1
)

In [ ]:
# Step 7) إعادة البناء / Reconstruct test images
reconstructed = autoencoder.predict(X_test, verbose=0)
test_mse = np.mean((X_test - reconstructed) ** 2)
print(f'Test reconstruction MSE: {test_mse:.6f}')

In [ ]:
# Step 8) original vs reconstructed / Visual comparison
n = 5
fig, axes = plt.subplots(2, n, figsize=(12, 4))
for i in range(n):
    axes[0, i].imshow(X_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray')
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')
plt.suptitle('Autoencoder: Original vs Reconstructed')
plt.tight_layout()
plt.show()

In [ ]:
# Step 9) فضاء latent / Latent space (first 2 dims for visualization)
latent_codes = encoder.predict(X_test[:100], verbose=0)
plt.figure(figsize=(5, 4))
plt.scatter(latent_codes[:, 0], latent_codes[:, 1], alpha=0.7, s=20)
plt.xlabel('Latent dim 1')
plt.ylabel('Latent dim 2')
plt.title('Latent Space (first 2 dimensions)')
plt.show()
print(f'Bottleneck compresses {input_dim} pixels -> {latent_dim} dimensions.')